# Kenyan voice design on VoxCPM2 - Colab LoRA

Fine-tunes **`openbmb/VoxCPM2`** so its *voice design* mode produces **Kenyan voices**.
VoxCPM2 already accepts a natural-language description in parentheses in front of the text:

```python
model.generate(text="(A Kenyan female voice in her thirties, mid-pitched)Habari yako?")
```

Out of the box that space is mostly English-accented. This notebook trains a LoRA on
`badrex/swahili-speech-400hr` (~380 Kenyan speakers, gender/age/location metadata) with
every transcript prefixed by a **truthful** description of that speaker's voice, so the
same kind of sentence designs a Kenyan voice in Swahili *and* in the English words
Kenyans mix in.

All heavy state (16 kHz wavs, caption manifests, base weights, checkpoints) lives on
**Google Drive**, so a disconnect costs at most one save interval:

> **After a disconnect:** Runtime > Run all. Finished stages are skipped and training
> resumes from the last checkpoint.

## 0. Runtime

Runtime > Change runtime type > Hardware accelerator:
- **L4** (Colab Pro, 24 GB) - recommended, ~20 GB is the LoRA footprint
- **A100** - faster
- **T4** - only with `--allow-small-gpu`, expect slow bf16 emulation

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
try:
    import torch
    print("torch", torch.__version__, "| cuda", torch.version.cuda,
          "| device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
except Exception as exc:
    print("torch check failed:", exc)

## 1. Drive + artifacts root

`ARTIFACTS` survives the VM. Resolution order is `--artifacts` > `$KENYA_ARTIFACTS` >
`/content/drive/MyDrive/kenya-voices` (when Drive is mounted) > `<repo>/artifacts`.

In [ ]:
from pathlib import Path

ARTIFACTS = "/content/drive/MyDrive/kenya-voices"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("not on Colab / Drive unavailable:", exc)

if Path("/content/drive/MyDrive").is_dir():
    Path(ARTIFACTS).mkdir(parents=True, exist_ok=True)
    print("artifacts ->", ARTIFACTS)
else:
    ARTIFACTS = str(Path.cwd() / "artifacts")
    print("WARNING: no Drive, artifacts are ephemeral ->", ARTIFACTS)

## 2. Code + install

Light deps only; torch stays as Colab shipped it. `kenya-voices env` later clones and
installs the VoxCPM repo with `--no-deps` plus a curated dependency list.

In [ ]:
REPO_URL = "https://github.com/lyle-mlengineer/audio-speech-ai.git"
DEST = Path("/content/audio-speech-ai")

if DEST.exists():
    !git -C {DEST} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {DEST}

%cd {DEST}/voice-design
!pip install -q -e .

## 3. Doctor

GPU / bf16 / VRAM, Drive, free space, VoxCPM install, base model, manifests.
Fix the FAILs (`env`, `data`, `model`) before training.

In [ ]:
import os

os.environ["KENYA_ARTIFACTS"] = ARTIFACTS  # survives every subprocess below
!kenya-voices doctor


## 4. Data: Kenyan speakers -> caption manifests

Defaults: 120 speakers x 30 clips, ~10 h audio, 3-20 s clips, 10% of **speakers** held
out for validation, ~10% caption dropout (so the adapter still works without a
description). Everything is deterministic and idempotent; rerunning reuses the wavs.

The streaming pass reads ~64k rows of metadata (no audio decoded), then filters audio to
the selected clips only - that is why this takes minutes, not an hour.

In [ ]:
# Inspect the data first (cheap, no GPU work).
!kenya-voices data --speakers 120 --clips-per-speaker 30 --hours 10 --caption-dropout 0.1
!head -c 600 {ARTIFACTS}/manifests/train.jsonl

## 5. One-shot pipeline

`all` = `env -> data -> model -> train -> eval -> export`, skipping anything already done.
**Re-run this exact cell after a disconnect.**

```text
--max-hours 8   stop gracefully before the Colab limit (trainer checkpoints on SIGINT)
--steps 1200    ~2 epochs over ~3.5k caption rows at effective batch 16
```

First run downloads ~4-5 GB of weights plus ~2 GB of audio; expect 20-40 min before the
first training step appears.

In [ ]:
!kenya-voices all --resume --max-hours 8

## 5b. Or drive the stages yourself

Same state, same resume behaviour, more control.

In [ ]:
# !kenya-voices env
# !kenya-voices data --speakers 120 --hours 10
# !kenya-voices model
# !kenya-voices train --max-hours 8
# !kenya-voices eval --ab
# !kenya-voices export

## 6. Status (this is what survived the disconnect)

In [ ]:
!kenya-voices status

## 7. Listen: does the caption design the voice?

`eval` writes one clip per design prompt at every saved step. The report scores **caption
adherence** objectively: a "male, low-pitched" caption must produce a lower median f0 than
the "female, higher-pitched" caption on the same text. With `--ab` the base model is
rendered too, so you can hear what the LoRA changed.

In [ ]:
from IPython.display import Audio, display, Markdown

run = "design_r32"
out = Path(ARTIFACTS, "evals", run)
report = out / "report.md"
if report.exists():
    display(Markdown(report.read_text()))

wavs = sorted(out.glob("step_*/*__lora.wav"))
print(f"{len(wavs)} LoRA clips")
for wav in wavs[-10:]:
    print(wav.relative_to(ARTIFACTS))
    display(Audio(str(wav)))

## 8. Design a voice from your own description

The trained adapter lives in `{ARTIFACTS}/exports/design_r32/step_*`. Captions that worked
during training are in `design_prompts.json` in the same folder.

Caption ingredients the data can actually support:

| Attribute | Example | Source |
|---|---|---|
| gender | `male`, `female` | dataset metadata |
| age | `in his thirties`, `in her early twenties`, `in her fifties or older` | dataset metadata |
| pitch | `low-pitched`, `mid-pitched`, `bright, higher-pitched` | measured median f0 |
| pace | `measured, unhurried pace`, `steady conversational pace`, `brisk city pace`, `fast, energetic delivery` | measured words/second |
| place | `from Nairobi`, `from Mombasa` | dataset metadata |
| language mix | `Swahili`, `Swahili with some English words`, `Swahili mixed with English (Sheng-style code-switching)` | measured English word share |

Anything beyond that (emotion, "warm", "authoritative", accent strength) is not in the
training labels - it may work, but the adapter was not taught it.

In [ ]:
import glob, shlex

exports = sorted(glob.glob(f"{ARTIFACTS}/exports/design_r32/step_*"))
LORA = exports[-1] if exports else ""

CAPTION = "A Kenyan female voice in her thirties, mid-pitched, steady conversational pace, Swahili mixed with English"
TEXT = "Habari yako! Leo tunaangalia mambo ya biashara na the new budget."

cmd = f"kenya-voices design --caption {shlex.quote(CAPTION)} --text {shlex.quote(TEXT)} --out /content/design.wav"
if LORA:
    cmd += f" --lora {shlex.quote(LORA)}"
print(cmd)
!{cmd}
display(Audio("/content/design.wav"))

## 9. Export + download the adapter

The package is portable: `lora_config.json` points at `openbmb/VoxCPM2`, not at a personal
Drive path, so anyone can load it with `lora_weights_path=<dir>`.

In [ ]:
import shutil

!kenya-voices export
exports = sorted(Path(ARTIFACTS, "exports", "design_r32").glob("step_*"))
if exports:
    bundle = shutil.make_archive("/content/kenyan_voice_design_lora", "zip", exports[-1])
    print("bundle:", bundle)
    from google.colab import files
    files.download(bundle)
else:
    print("nothing exported yet - run the pipeline first")

## Troubleshooting

| Symptom | Fix |
|---|---|
| `No CUDA GPU` | Runtime > Change runtime type > GPU, then re-run cell 0 |
| `GPU is not ideal ... VRAM < 20 GB` | You are on a T4: `!kenya-voices train --profile t4 --allow-small-gpu` |
| CUDA OOM during training | next profile down, or `!kenya-voices train --set max_batch_tokens=4096` |
| `sample_rate mismatch` | never edit the rendered YAML: AudioVAE encodes at 16 kHz |
| Data build is slow | it streams the HF dataset; lower `--hours 5 --speakers 60`, or rerun (wavs on Drive are reused) |
| Valueless captions ("A Kenyan speaker") | the clip had no gender metadata and too few f0 measurements; normal for a handful of speakers |
| Training replays training audio | overfit: use an earlier `step_*` checkpoint (`eval --steps`), or raise `--caption-dropout` |
| Disconnect | Runtime > Run all. `all --resume` skips finished stages and resumes training |